> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🧑‍🏫 수행평가 채점 도우미 (교사용)

학생용 노트북(10_assessment_project)과 **같은 함수**로 나만의 조건을 만들어요. 자세한 채점 기준과 구술 확인 질문은 docs/ASSESSMENT.md에 있어요.

1. **조건표** — 학번 목록을 넣으면 학생마다의 SEED·TEST_SIZE·꼭 할 실험·확인 행이 나와요. 학생 노트북이 이 조건을 그대로 썼는지 견줘요.
2. **지문 확인** — 학생 노트북 맨 아래의 요약코드와 지문을 붙여 넣으면, 손으로 고친 흔적이 없는지와 조건이 학번과 맞는지 확인해요.
3. **점수 합계** — 평가 요소별 점수를 적으면 합계와 채점표 CSV가 만들어져요.

> 의심스러운 노트북은 학생 사본을 열어 **런타임 → 모두 실행**을 해 보세요. 같은 지문이 나오면 그 노트북의 결과가 맞아요.

- 학번마다의 나만의 조건을 표로 만들어요. 인쇄해 두면 순회하며 학생 화면과 바로 견줄 수 있어요.

In [ ]:
import pandas as pd

import hashlib, json

실험과제_목록 = [
    '입력 속성 하나를 빼고 다시 학습해, 그 속성이 성능에 얼마나 도움이 되는지 비교하기',
    '하이퍼파라미터(knn은 k, tree는 max_depth)를 세 가지 값으로 바꿔 성능 비교하기',
    '정규화(StandardScaler)를 하기 전과 한 뒤의 성능 비교하기',
    '훈련 데이터를 절반만 써서 학습해 성능 비교하기',
    '다른 알고리즘 하나로 바꿔 성능 비교하기',
]

def 나만의_조건(학번):
    h = int(hashlib.sha256(('ai-project:' + str(학번).strip()).encode('utf-8')).hexdigest(), 16)
    return {
        'SEED': h % 10000,
        'TEST_SIZE': [0.2, 0.25, 0.3][(h // 1000) % 3],
        '실험과제': 실험과제_목록[(h // 3000) % len(실험과제_목록)],
        '확인행': (h // 15000) % 5,
    }

def 지문_만들기(요약):
    글 = json.dumps(요약, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(글.encode('utf-8')).hexdigest()[:8]

학번목록 = ['10101', '10102', '10103']   # ✏️ 우리 반 학번

조건표 = pd.DataFrame([{'학번': h, **나만의_조건(h)} for h in 학번목록])
조건표

- 지문은 요약코드 글자에서 만들어요. 한 글자라도 바뀌면 지문이 달라져요.
- 꼭 할 실험을 했는지는 실험 목록의 "바꾼 것"을 보고 판단해요.

In [ ]:
요약코드 = ''   # ✏️ 학생 노트북의 "요약코드:" 뒤의 글을 그대로 붙여 넣어요 (중괄호 { } 까지)
지문 = ''       # ✏️ 학생 노트북의 "🔏 지문:" 뒤의 여덟 글자

if 요약코드.strip():
    요약 = json.loads(요약코드)
    맞음 = 지문_만들기(요약) == 지문.strip()
    print('🔏 지문', '✅ 일치 — 요약코드를 손으로 고치지 않았어요' if 맞음 else '⚠️ 다름 — 요약코드나 지문이 바뀌었어요. 학생 사본을 모두 실행해 확인하세요')
    기대 = 나만의_조건(요약['학번'])
    for 키 in ['SEED', 'TEST_SIZE']:
        print(f'🔑 {키}', '✅' if 요약[키] == 기대[키] else f'⚠️ 학번의 조건은 {기대[키]}인데 {요약[키]}을(를) 썼어요')
    print(f'🧪 실험 {len(요약["실험"])}개 · 꼭 할 실험: {기대["실험과제"]}')
    for e in 요약['실험']:
        print('   ·', e)
else:
    print('요약코드를 붙여 넣고 실행하세요.')

- 평가 요소마다 수준별 점수(① 30·20·10, ② 30·20·10, ③ 25·15·5, ④ 15·10·5)를 적어요. 가장 낮은 수준의 합이 기본점수 30점이에요.
- 구술 확인에서 ×이면 관련 요소를 한 단계 낮춰 적어요(docs/ASSESSMENT.md).

In [ ]:
# ✏️ 한 줄에 학생 한 명: 학번, 이름, ①(30·20·10), ②(30·20·10), ③(25·15·5), ④(15·10·5), 구술(○·△·×), 메모
# 낼 뜻이 없어 내지 않은 학생은 ①~④를 None으로 두면 20점(미응시)이 돼요.
채점 = [
    ['10101', '김펭귄', 30, 20, 15, 10, '○', ''],
    ['10102', '이아델', None, None, None, None, '', '미제출'],
]

def 합계(행):
    점수 = 행[2:6]
    if any(s is None for s in 점수):
        return 20
    return sum(점수)

채점표 = pd.DataFrame(채점, columns=['학번', '이름', '①문제정의', '②데이터·구현', '③성능·개선', '④표현·설명', '구술', '메모'])
채점표['합계'] = [합계(r) for r in 채점]
채점표.to_csv('채점표.csv', index=False, encoding='utf-8-sig')
print('💾 채점표.csv를 만들었어요 (왼쪽 📁 파일에서 내려받기)')
채점표